Modelo

In [2]:
from models.model_CRNN_ConvNeXt_Tiny_BiLSTM import Model

In [ ]:
from models.model_CRNN_ConvNeXt_Tiny_BiLSTM import Model
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

from dataset import avesDataset, SpectrogramAugment, TransformedSubset
from train import train_model


data_dir = "../dataset"
full_dataset = avesDataset(data_dir)

# Pega o número real de classes do dataset
num_classes = len(full_dataset.label_map)

# Instancia o modelo com o número correto de classes
model = Model(num_classes=num_classes)

# ──────────────────────────────────────────────────────────────────────────────
# Tópico 2: Split estratificado por label (evita data leakage por classe e
# garante que cada fold tenha a mesma proporção de cada espécie).
# ──────────────────────────────────────────────────────────────────────────────
all_indices = list(range(len(full_dataset)))
all_labels  = [full_dataset.samples[i][1] for i in all_indices]

# 80% treino | 20% temp
train_idx, temp_idx, _, temp_labels = train_test_split(
    all_indices, all_labels,
    test_size=0.2,
    stratify=all_labels,
    random_state=42
)

# 50% do temp → val  |  50% do temp → test  (= 10% / 10% do total)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

print(f"[INFO] Split estratificado — treino: {len(train_idx)} | val: {len(val_idx)} | teste: {len(test_idx)}")

train_subset = Subset(full_dataset, train_idx)
val_subset   = Subset(full_dataset, val_idx)
test_subset  = Subset(full_dataset, test_idx)

# Augmentation apenas no treino
train_dataset = TransformedSubset(train_subset, transform=SpectrogramAugment())
val_dataset   = val_subset
test_dataset  = test_subset

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

train_model(
    model=model,
    num_epochs=50,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    patience=10,
    min_delta=0.0001,
    lr=0.0001
)


[INFO] Split estratificado — treino: 5755 | val: 719 | teste: 720
Usando dispositivo: cuda
[INFO] Detectadas 18 classes dinamicamente.
Epoca [1/50] - Loss Treino: 2.2793, Acc Treino: 32.72% | Loss Val: 1.6950, Acc Val: 54.94%
[OK] Melhor modelo atualizado!
Epoca [2/50] - Loss Treino: 1.6732, Acc Treino: 57.71% | Loss Val: 1.4801, Acc Val: 64.81%
[OK] Melhor modelo atualizado!
Epoca [3/50] - Loss Treino: 1.5361, Acc Treino: 63.65% | Loss Val: 1.3966, Acc Val: 68.01%
[OK] Melhor modelo atualizado!
Epoca [4/50] - Loss Treino: 1.4604, Acc Treino: 66.27% | Loss Val: 1.3404, Acc Val: 69.40%
[OK] Melhor modelo atualizado!
Epoca [5/50] - Loss Treino: 1.3776, Acc Treino: 69.17% | Loss Val: 1.3317, Acc Val: 71.77%
--- Checkpoint salvo em: result\CRNN_ConvNeXt_Tiny_BiLSTM_2\checkpoint_epoch_5.pth
[OK] Melhor modelo atualizado!
Epoca [6/50] - Loss Treino: 1.3582, Acc Treino: 70.50% | Loss Val: 1.3072, Acc Val: 71.35%
[OK] Melhor modelo atualizado!
Epoca [7/50] - Loss Treino: 1.3209, Acc Treino: 71

(Model(
   (backbone): ConvNeXtExtractor(
     (features): Sequential(
       (0): Conv2dNormActivation(
         (0): Conv2d(1, 96, kernel_size=(4, 4), stride=(4, 4))
         (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
       )
       (1): Sequential(
         (0): CNBlock(
           (block): Sequential(
             (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
             (1): Permute()
             (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
             (3): Linear(in_features=96, out_features=384, bias=True)
             (4): GELU(approximate='none')
             (5): Linear(in_features=384, out_features=96, bias=True)
             (6): Permute()
           )
           (stochastic_depth): StochasticDepth(p=0.0, mode=row)
         )
         (1): CNBlock(
           (block): Sequential(
             (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
             (1): Permute()
    